# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 dataset using the `mlcroissant` library. Key steps include: loading the Croissant schema metadata, reviewing record sets and fields by their `@id`s, extracting data, and performing basic analysis and visualizations.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print key metadata
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Authors: {len(metadata.author)} - See `metadata.author` list for `@id`s.")
print(f"Identifier: {metadata.identifier}")
print(f"Published Date: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, their fields, and corresponding `@id`s.

In Croissant datasets, each record set and its fields/columns are referenced by a unique `@id`. We'll display the record sets and their field IDs for inspection.

In [ ]:
# Find available record sets (by @id)
record_sets = dataset.metadata.recordSet
if not record_sets:
    # Check for default or implicit record set in the distribution
    print("No explicit recordSet found; attempting to discover from distribution.")
    # mlcroissant often infers data from the first distribution
    inferred_record_set_id = 'cr:recordSet/frontiers/7862866/629c16ec-37ee-4556-a351-d5164116c2dd-1'
    record_sets = [inferred_record_set_id]
else:
    record_sets = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in record_sets]

print("Record sets found by @id:")
for rid in record_sets:
    print(f"- {rid}")
    # List fields for this record set - get the field IDs
    try:
        record_set_obj = dataset.metadata.find_by_id(rid)
        fields = record_set_obj.field if hasattr(record_set_obj, 'field') else []
        field_ids = [field['@id'] if isinstance(field, dict) else field for field in fields]
        print("  Fields:")
        for fid in field_ids:
            print(f"    - {fid}")
    except Exception as e:
        print("  (Could not enumerate fields; error:", e, ")")

## 3. Data Extraction
Load data from each available record set into pandas DataFrames for analysis.

Below, we reference each record set by its `@id`, and extract their records. The column names correspond to each field's `@id` as defined in the Croissant schema.

In [ ]:
# Prepare record sets list
record_sets_to_use = record_sets  # From previous cell
dataframes = {}

for record_set_id in record_sets_to_use:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"DataFrame for record set {record_set_id}:")
    print(f"Columns (@id): {df.columns.tolist()}")
    print(df.head(3), "\n")

# For further exploration, select the first record set
main_record_set_id = record_sets_to_use[0]
main_df = dataframes[main_record_set_id]

## 4. Exploratory Data Analysis (EDA)
We'll perform basic EDA using columns/fields referenced by their `@id`. Example steps: filtering on numeric fields, normalization, grouping, and summarization.

**Hint:** Numeric and categorical fields can be identified using field `@id`s from the overview above. We'll select a numeric field for filtering and normalization, and group by a categorical field.

In [ ]:
# Identify numeric and categorical fields (by @id)
numeric_field = None
group_field = None

# Try to find likely numeric/categorical fields by inspecting columns
print(f"Available columns (@id): {main_df.columns.tolist()}")

# Example usage: Assuming there are typical fields like 'cr:field/Age' or 'cr:field/IntervalBetweenDiagnoses'
# Replace these with actual @id names when known
possible_numeric_fields = [col for col in main_df.columns if 'age' in col.lower() or 'interval' in col.lower() or 'number' in col.lower()]
possible_group_fields = [col for col in main_df.columns if 'sex' in col.lower() or 'msi' in col.lower() or 'location' in col.lower()]

# Select
numeric_field = possible_numeric_fields[0] if possible_numeric_fields else main_df.columns[0]
group_field = possible_group_fields[0] if possible_group_fields else main_df.columns[1]

print(f"Using numeric field: {numeric_field}")
print(f"Using group/categorical field: {group_field}")

# Filtering numeric field: remove outliers (e.g. records where value > threshold)
threshold = 10  # Adjust as appropriate
if pd.api.types.is_numeric_dtype(main_df[numeric_field]):
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
else:
    filtered_df = main_df.copy()  # If not numeric, skip filtering

print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize numeric field
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by group_field (e.g., MSI status, Sex, Location, etc.)
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped mean of {numeric_field} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields.

Here, we'll create a histogram of the selected numeric field and a bar plot of group counts.

In [ ]:
# Histogram of numeric field
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
    plt.figure(figsize=(6, 4))
    filtered_df[numeric_field].hist(bins=10)
    plt.title(f"Distribution of {numeric_field} (by @id)")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

# Bar plot of group counts
if group_field in filtered_df.columns:
    plt.figure(figsize=(6, 4))
    filtered_df[group_field].value_counts().plot(kind='bar')
    plt.title(f"Counts by {group_field} (by @id)")
    plt.xlabel(group_field)
    plt.ylabel("Count")
    plt.show()

## 6. Conclusion
This notebook illustrated how to use `mlcroissant` to load and explore a FAIR^2 Croissant dataset, referencing all entities by their `@id`. We've shown how to find record sets and fields, filter and normalize numeric data, group by categorical attributes, and visualize distributions.

**Key observations:**
- The dataset contains rich clinical variables for second primary colorectal cancer in survivors.
- Data can be accessed and manipulated easily by referencing fields using their `@id`.
- Exploratory analysis reveals basic trends and allows grouping/aggregation for deeper insight.

For further analysis, consult the schema's field definitions and documentation for more precise mappings of medical concepts.

### References
- [mlcroissant Documentation](https://mlcroissant.org/docs/)
- [FAIR^2 Dataset: Clinical CRC Survivors](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)
